# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5C3EE8?style=flat-square&logo=opencv&logoColor=white)](https://opencv.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 4 — Evaluación de Image Enhancement (Retinex, EndoLMSPEC e IAT) sobre Estimación de Profundidad Endoscópica en SCARED</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### Objetivos de este notebook

Evaluar si los modelos de **Image Enhancement** mejoran el desempeño de Endo-Depth en estimación de profundidad sobre SCARED:

1. Comparar **cuatro condiciones de preprocesamiento**: sin corrección (baseline), Retinex SSR, EndoLMSPEC e IAT
2. Cuantificar el impacto en métricas estándar de depth: **AbsRel, RMSE, Chamfer Distance**
3. Analizar si la mejora se concentra en **zonas especulares** (hipótesis central del proyecto)
4. Evaluar el **costo computacional** de cada corrección para determinar viabilidad clínica en tiempo real

> García-Vega, A., et al. (2022). Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging. *arXiv:2210.15033*.
>
> Rahman, Z., Jobson, D. J., & Woodell, G. A. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1), 100–110. https://doi.org/10.1117/1.1636183
>
> Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement: A Benchmark and Transformer-Based Method. *AAAI 2022*.

---
## 0. Pipeline general del experimento

El siguiente diagrama muestra el flujo completo del experimento. La variable independiente es el **método de image enhancement** aplicado antes de la inferencia de profundidad. Todo lo demás (modelo de depth, dataset, métricas) permanece constante.

```mermaid
flowchart TD
    A["🗂️ SCARED Dataset\ndatasets 8–9, keyframes 0–4\nLeft_Image.png + depth GT (mm)"] --> B

    B["📷 Imagen RGB original\n1280×1024 px, uint8"] --> C1 & C2 & C3 & C4

    subgraph ENHANCEMENT [" ⚙️ Variable Independiente: Image Enhancement "]
        C1["❌ None\n(baseline)"]
        C2["🔆 Retinex SSR\nσ=30\nRahman et al. 2004"]
        C3["🧠 EndoLMSPEC\nU-Net + Laplaciana\nGarcía-Vega et al. 2022"]
        C4["⚡ IAT\nIllumination-Adaptive Transformer\nWang et al. 2022"]
    end

    C1 & C2 & C3 & C4 --> D

    D["🔍 Endo-Depth\nResNet18 + DepthDecoder\nRecasens et al. 2021"] --> E

    E["📐 Median Scaling → mm"] --> F

    subgraph METRICS [" 📊 Métricas "]
        F1["AbsRel"] 
        F2["RMSE (mm)"]
        F3["Chamfer (mm)"]
        F4["AbsRel especular"]
        F5["PSNR / SSIM"]
        F6["FPS"]
    end

    F --> F1 & F2 & F3 & F4 & F5 & F6

    style ENHANCEMENT fill:#fff8e1,stroke:#E0A800,stroke-width:2px
    style METRICS fill:#e8f4fd,stroke:#2766CB,stroke-width:2px
```

### Hipótesis

Los modelos de image enhancement reducen la perturbación fotométrica causada por **reflejos especulares** y **gradientes de iluminación**, mejorando la estimación de profundidad monocular de Endo-Depth. La mejora debería ser mayor en zonas especulares que en el resto de la imagen.

In [ ]:
import subprocess, sys

# Solo lo que Colab no trae por defecto
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tifffile", "scikit-image"])

import torch, tifffile, cv2, numpy as np
print(f"torch    : {torch.__version__}")
print(f"tifffile : {tifffile.__version__}")
print(f"opencv   : {cv2.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")

---
## 1. Configuración de rutas

### Estructura esperada en Google Drive

```
MyDrive/proyecto_integrador/
├── scared_raw/                    ← datasets SCARED (zips)
├── endo_depth_weights/            ← encoder.pth + depth.pth
├── Endo-Depth-and-Motion/         ← código del modelo
├── EndoLMSPEC/                    ← repo EndoLMSPEC
│   └── checkpoint/main_net/
│       └── model_256_combined_SSIM5_1.pth
├── EndoVit/                       ← repo IAT (mal nombrado, contiene IAT)
│   └── experiments/
│       ├── model/IAT_main.py
│       └── best_Epoch_exposure.pth
└── avance4_outputs/               ← resultados (se crea automáticamente)
```

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    MODEL_PATH   = BASE / "endo_depth_weights"
    SCARED_ROOT  = BASE / "scared_raw"
    EDAM_PATH    = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH  = BASE / "EndoLMSPEC"
    IAT_PATH     = BASE / "EndoVit" / "experiments"
    OUT_DIR      = BASE / "avance4_outputs"
else:
    MODEL_PATH   = Path("E:/endo_depth_weights")
    SCARED_ROOT  = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH    = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH  = Path("E:/EndoLMSPEC")
    IAT_PATH     = Path("E:/EndoVit/experiments")
    OUT_DIR      = Path("../outcomes/avance4_enhancement")

LMSPEC_WEIGHTS = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS    = IAT_PATH / "best_Epoch_exposure.pth"

OUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_KEYFRAMES = [
    ("dataset_8", "keyframe_0"),
    ("dataset_8", "keyframe_1"),
    ("dataset_8", "keyframe_2"),
    ("dataset_8", "keyframe_3"),
    ("dataset_8", "keyframe_4"),
    ("dataset_9", "keyframe_0"),
    ("dataset_9", "keyframe_1"),
    ("dataset_9", "keyframe_2"),
    ("dataset_9", "keyframe_3"),
    ("dataset_9", "keyframe_4"),
]
CAP_MM = 150.0

print(f"Entorno       : {'Google Colab' if IN_COLAB else 'Local'}")
for name, p in [("MODEL_PATH", MODEL_PATH), ("SCARED_ROOT", SCARED_ROOT),
                ("EDAM_PATH", EDAM_PATH), ("LMSPEC_PATH", LMSPEC_PATH),
                ("LMSPEC_WEIGHTS", LMSPEC_WEIGHTS),
                ("IAT_PATH", IAT_PATH), ("IAT_WEIGHTS", IAT_WEIGHTS)]:
    status = "✓" if p.exists() else "✗ NO ENCONTRADO"
    print(f"  {name:16s}: {status}")

---
## 2. Cargar Endo-Depth

Mismo modelo que en Avances 3 y 4: ResNet18 encoder + DepthDecoder, pesos Hamlyn. No se modifica el modelo — es la variable dependiente fija del experimento.

In [ ]:
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

encoder = ResnetEncoder(18, False)
loaded_enc = torch.load(MODEL_PATH / "encoder.pth", map_location=DEVICE)
FEED_HEIGHT = loaded_enc["height"]
FEED_WIDTH  = loaded_enc["width"]
filtered_enc = {k: v for k, v in loaded_enc.items() if k in encoder.state_dict()}
encoder.load_state_dict(filtered_enc)
encoder.to(DEVICE).eval()

depth_decoder = DepthDecoder(num_ch_enc=encoder.num_ch_enc, scales=range(4))
loaded_dec = torch.load(MODEL_PATH / "depth.pth", map_location=DEVICE)
depth_decoder.load_state_dict(loaded_dec)
depth_decoder.to(DEVICE).eval()

print(f"Resolución del modelo: {FEED_HEIGHT}×{FEED_WIDTH}")
print("Endo-Depth cargado ✓")

---
## 3. EndoLMSPEC — Arquitectura y carga del modelo

### 3.1 ¿Qué es EndoLMSPEC?

EndoLMSPEC (*Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging*, García-Vega et al., 2022) es una extensión del método LMSPEC de Afifi et al. (2021) adaptada específicamente para imágenes endoscópicas. Resuelve el problema de **corrección de exposición** (tanto sobreexposición por reflejos especulares como subexposición en zonas periféricas) sin datos pareados anotados manualmente — aprende a corregir usando la distribución estadística de imágenes endoscópicas normales.

Fue entrenado en **Endo4IE**, el primer dataset endoscópico diseñado específicamente para evaluación de métodos de image enhancement, que contiene frames reales y sintéticos con exposición incorrecta y sus correspondientes imágenes de referencia.

### 3.2 Arquitectura detallada

```mermaid
flowchart LR
    IN["Imagen de entrada\nHxWx3 float32"] --> LP

    subgraph LP [" 🔺 Pirámide Laplaciana (4 niveles) "]
        direction TB
        L4["Level 4\nH/8 × W/8\n(low-freq)"]
        L3["Level 3\nH/4 × W/4"]
        L2["Level 2\nH/2 × W/2"]
        L1["Level 1\nH × W\n(high-freq)"]
    end

    L4 --> U1

    subgraph UNETS [" 🧱 Cascada de U-Nets "]
        direction TB
        U1["UNet24\n(sin residual)\nCorrige low-freq"]
        U2["UNet24-res\n(+ residual)\nRefina freq. medias"]
        U3["UNet24-res\n(+ residual)\nRefina más detalle"]
        U4["UNet16-res\n(+ residual)\nReconstrucción final"]
    end

    U1 -->|"y_hat0 + L3"| U2
    U2 -->|"y_hat1 + L2"| U3
    U3 -->|"y_hat2 + L1"| U4

    U4 --> OUT["subnet_16\nImagen corregida\nHxWx3"]

    style LP fill:#fff8e1,stroke:#E0A800
    style UNETS fill:#e8f4fd,stroke:#2766CB
```

**Clave de la arquitectura**: la imagen de entrada se descompone en una **pirámide Laplaciana** de 4 niveles. El nivel 4 (más pequeño, baja frecuencia) captura la iluminación global; los niveles superiores capturan detalles de alta frecuencia (texturas, bordes, venas). Cada U-Net procesa un nivel de la pirámide y pasa su resultado al siguiente nivel sumando con los detalles de alta frecuencia. Esto permite corregir la iluminación global sin destruir la textura del tejido.

### 3.3 Pirámide Laplaciana

La pirámide Laplaciana descompone la imagen como:

$$L_k = G_k - \text{pyrUp}(G_{k+1})$$

donde $G_k$ es el nivel $k$ de la pirámide Gaussiana. Cada $L_k$ contiene las **diferencias** entre escalas consecutivas — esencialmente el contenido de alta frecuencia a esa escala. La imagen original se puede reconstruir exactamente sumando todos los niveles de la pirámide Laplaciana.

**¿Por qué es ideal para endoscopía?** Los reflejos especulares son eventos de alta frecuencia localizada (brillos puntuales), mientras que el gradiente radial de iluminación del endoscopio es de baja frecuencia. La pirámide Laplaciana los separa naturalmente, permitiendo que la red corrija la iluminación global (niveles bajos) sin alterar los detalles del tejido (niveles altos).

### 3.4 Función de pérdida

EndoLMSPEC usa una pérdida compuesta de 4 términos:

$$\mathcal{L} = \alpha \mathcal{L}_{rec} + \beta \mathcal{L}_{pyr} + \gamma \mathcal{L}_{SSIM} + \delta \mathcal{L}_{adv}$$

donde:
- $\mathcal{L}_{rec}$: reconstrucción pixel-wise (L1)
- $\mathcal{L}_{pyr}$: consistencia de pirámide Laplaciana
- $\mathcal{L}_{SSIM}$: similitud estructural (aportación principal vs. LMSPEC original)
- $\mathcal{L}_{adv}$: pérdida adversarial (discriminador)

La inclusión de $\mathcal{L}_{SSIM}$ es la innovación principal respecto al LMSPEC original: fuerza a la red a preservar la **estructura del tejido** (bordes, venas, pliegues) incluso al corregir la exposición — esencial para que la imagen corregida sea útil para estimación de profundidad.

In [ ]:
import sys
import torch
import numpy as np
import torchvision.transforms as T
import PIL.Image as pil

# Agregar EndoLMSPEC al path para importar Generator
sys.path.insert(0, str(LMSPEC_PATH))
from generator import Generator

lmspec_net = Generator(n_channels=3, device=DEVICE, bilinear=False)
lmspec_net.load_state_dict(
    torch.load(LMSPEC_WEIGHTS, map_location=DEVICE)
)
lmspec_net.to(DEVICE).eval()

n_params = sum(p.numel() for p in lmspec_net.parameters())
print(f"EndoLMSPEC parámetros: {n_params/1e6:.2f} M")
print(f"Pesos cargados desde: {LMSPEC_WEIGHTS.name}")
print("EndoLMSPEC cargado ✓")

---
## 3b. IAT — Illumination-Adaptive Transformer

### ¿Qué es IAT?

IAT (*Illumination-Adaptive Transformer*, Wang et al., 2022) es un método de corrección de iluminación basado en transformers que opera en dos ramas paralelas:

- **Rama local** (`Local_pred_S`): red convolucional ligera (3 bloques Conv+LN por rama) que predice dos mapas por píxel — un factor multiplicativo `mul` y un desplazamiento aditivo `add`. La imagen corregida localmente es: $\hat{I}_{local} = I \odot mul + add$
- **Rama global** (`Global_pred`): un Swin Transformer que predice una curva gamma $\gamma$ y una matriz de corrección de color (CCM) 3×3 aplicadas globalmente sobre $\hat{I}_{local}$

```mermaid
flowchart LR
    IN["Imagen\n[0,1] float"] --> LP & GP

    subgraph LP [" 🔲 Rama Local "]
        LC["Conv + 3×CBlock_ln"] --> MUL["mul map\n(ReLU)"]
        LC --> ADD["add map\n(Tanh)"]
    end

    subgraph GP [" 🌐 Rama Global "]
        ST["Swin Transformer\n(Global_pred)"] --> GAMMA["γ (gamma)"]
        ST --> CCM["CCM 3×3\n(color matrix)"]
    end

    IN --> MULT["I × mul + add\n= I_local"]
    MUL & ADD --> MULT
    MULT --> APPLY["apply_color(I_local, CCM)^γ"]
    GAMMA & CCM --> APPLY
    APPLY --> OUT["Imagen corregida\n[0,1] float"]

    style LP fill:#e8f4fd,stroke:#2766CB
    style GP fill:#fff8e1,stroke:#E0A800
```

**Ventaja sobre Retinex y EndoLMSPEC**: IAT es extremadamente ligero (~90K parámetros) y rápido. La corrección global con CCM y gamma es especialmente útil para imágenes con dominancias de color (como el tono rojizo/marrón del tejido endoscópico con sangre), mientras que la corrección local adapta el contraste píxel a píxel.

> Wang, T., Zhang, K., Shen, T., Luo, W., Stenger, B., & Lu, T. (2022). Ultra-High-Definition Low-Light Image Enhancement: A Benchmark and Transformer-Based Method. *AAAI 2022*.

In [ ]:
import sys
import torch

# IAT requiere timm (disponible en Colab)
sys.path.insert(0, str(IAT_PATH))
from model.IAT_main import IAT

iat_net = IAT(in_dim=3, with_global=True, type='lol')
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE))
iat_net.to(DEVICE).eval()

n_params = sum(p.numel() for p in iat_net.parameters())
print(f"IAT parámetros : {n_params/1e3:.1f} K  (muy ligero)")
print(f"Pesos cargados : {IAT_WEIGHTS.name}")
print("IAT cargado ✓")

---
## 4. Métodos de Image Enhancement

### 4.1 Baseline: sin corrección
La imagen original sin ningún preprocesamiento. Referencia para cuantificar el impacto de los métodos.

### 4.2 Retinex SSR
$$R_{SSR}(x,y) = \log I(x,y) - \log\left[G_\sigma * I(x,y)\right], \quad \sigma=30$$
> Rahman, Z., Jobson, D. J., & Woodell, G. A. (2004). *Journal of Electronic Imaging*, 13(1), 100–110. https://doi.org/10.1117/1.1636183

### 4.3 EndoLMSPEC
Pirámide Laplaciana de 4 niveles → cascada de 4 U-Nets con pérdida SSIM. Salida: `subnet_16` a resolución completa. Ver sección 3 para detalles.

### 4.4 IAT (Illumination-Adaptive Transformer)
Corrección local (mul/add por píxel vía conv) + corrección global (gamma + CCM vía Swin Transformer). ~90K parámetros, muy rápido. Ver sección 3b para detalles.

> Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.

In [ ]:
import cv2
import numpy as np
import torch
import torchvision.transforms as T


def correct_none(img_rgb: np.ndarray) -> np.ndarray:
    return img_rgb


def correct_retinex(img_rgb: np.ndarray, sigma: float = 30) -> np.ndarray:
    """Single-Scale Retinex (Rahman et al., 2004)."""
    img_f = img_rgb.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:, :, c], (0, 0), sigma)
        result[:, :, c] = np.log(img_f[:, :, c]) - np.log(blur + 1.0)
    result -= result.min()
    return (result / (result.max() + 1e-8) * 255).astype(np.uint8)


def correct_endolmspec(img_rgb: np.ndarray, net, device) -> np.ndarray:
    """EndoLMSPEC: pirámide Laplaciana + U-Nets (García-Vega et al., 2022)."""
    img_t = T.ToTensor()(img_rgb).to(device)
    with torch.no_grad():
        _, outputs = net(img_t)
    out = outputs['subnet_16'][0].cpu().clamp(0, 1)
    return (out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def correct_iat(img_rgb: np.ndarray, net, device) -> np.ndarray:
    """IAT: local mul/add + global gamma+CCM (Wang et al., 2022)."""
    # IAT espera [0,1] float, sin normalización adicional
    img_f = img_rgb.astype(np.float32) / 255.0
    img_t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        _, _, enhanced = net(img_t)
    out = enhanced[0].cpu().clamp(0, 1)
    return (out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


CORRECTIONS = {
    "none":       lambda img: correct_none(img),
    "retinex":    lambda img: correct_retinex(img),
    "endolmspec": lambda img: correct_endolmspec(img, lmspec_net, DEVICE),
    "iat":        lambda img: correct_iat(img, iat_net, DEVICE),
}

print("Funciones de corrección definidas ✓")
print(f"Métodos: {list(CORRECTIONS.keys())}")

---
## 5. Carga de datos SCARED

Mismo protocolo que Avance 3: datasets 8–9 (split de test, animal distinto al de train). Cada keyframe contiene `Left_Image.png` y `left_depth_map.tiff` (canal Z en mm).

In [ ]:
import io, zipfile, tifffile
import cv2
import numpy as np
import matplotlib.pyplot as plt


def load_keyframe(scared_root, dataset_id, keyframe_id):
    zip_path = scared_root / f"{dataset_id}.zip"
    with zipfile.ZipFile(zip_path) as z:
        buf = np.frombuffer(
            z.read(f"{dataset_id}/{keyframe_id}/Left_Image.png"), np.uint8)
        img = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        with z.open(f"{dataset_id}/{keyframe_id}/left_depth_map.tiff") as f:
            tiff = tifffile.imread(io.BytesIO(f.read()))
        depth_z = tiff[..., 2].astype(np.float32)
        depth_z[depth_z <= 0] = np.nan
    return img, depth_z


# Diagnóstico con primer keyframe
img_test, depth_test = load_keyframe(SCARED_ROOT, *EVAL_KEYFRAMES[0])
print(f"Imagen : {img_test.shape}  dtype={img_test.dtype}")
print(f"Depth  : {depth_test.shape}  rango=[{np.nanmin(depth_test):.1f}, {np.nanmax(depth_test):.1f}] mm")
print(f"GT válido: {(~np.isnan(depth_test)).mean()*100:.1f}%")

---
## 6. Visualización comparativa de los métodos de enhancement

Antes de correr el experimento completo, comparamos visualmente los tres métodos sobre el keyframe de prueba. Esto permite verificar que:

1. EndoLMSPEC corrige correctamente la exposición
2. Retinex elimina el gradiente de iluminación
3. Ambos preservan la textura del tejido

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
labels = {"none": "Sin corrección (baseline)",
          "retinex": "Retinex SSR (σ=30)\nRahman et al. 2004",
          "endolmspec": "EndoLMSPEC\nGarcía-Vega et al. 2022"}

for col, (name, fn) in enumerate(CORRECTIONS.items()):
    t0 = time.perf_counter()
    corrected = fn(img_test)
    t_ms = (time.perf_counter() - t0) * 1000

    axes[0, col].imshow(corrected)
    axes[0, col].set_title(f"{labels[name]}\n({t_ms:.0f} ms)", fontsize=10)
    axes[0, col].axis("off")

    # Diferencia vs. original
    diff = np.abs(img_test.astype(np.float32) - corrected.astype(np.float32)).mean(axis=2)
    im = axes[1, col].imshow(diff, cmap="hot", vmin=0, vmax=50)
    axes[1, col].set_title(f"Δ vs. original\nmedio={diff.mean():.1f} DN", fontsize=10)
    axes[1, col].axis("off")
    plt.colorbar(im, ax=axes[1, col], shrink=0.8)

plt.suptitle(f"Comparativa de métodos de enhancement — {EVAL_KEYFRAMES[0][0]}/{EVAL_KEYFRAMES[0][1]}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "avance5_enhancement_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Pipeline de inferencia y métricas

### 7.1 Inferencia con Endo-Depth

Para cada imagen corregida:
1. Resize a resolución de entrenamiento del modelo
2. Forward pass por encoder ResNet18 + DepthDecoder
3. Interpolación bilineal a resolución original
4. Conversión disparidad → profundidad relativa
5. **Median scaling** para convertir a mm

### 7.2 Métricas

| Métrica | Fórmula | Unidad | Qué mide |
|---|---|---|---|
| **AbsRel** | $\frac{1}{N}\sum\frac{|d-d^*|}{d^*}$ | adim | Error relativo medio — métrica primaria |
| **RMSE** | $\sqrt{\frac{1}{N}\sum(d-d^*)^2}$ | mm | Error cuadrático — penaliza errores grandes |
| **Chamfer** | $\frac{1}{2}(\overline{d_{P\to Q}} + \overline{d_{Q\to P}})$ | mm | Similitud de nubes 3D |
| **AbsRel_spec** | AbsRel en zona especular | adim | Efecto en zonas brillantes |
| **AbsRel_nospec** | AbsRel fuera de especular | adim | Efecto en tejido normal |
| **PSNR** | $10\log_{10}(255^2/\text{MSE})$ | dB | Fidelidad visual de la imagen corregida |
| **SSIM** | similitud estructural | [0,1] | Preservación de estructura visual |

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import PIL.Image as pil
from torchvision import transforms
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

FX, FY = 1078.0, 1078.0
CX, CY = 640.0,  512.0


def predict_depth(img_rgb, encoder, decoder, feed_h, feed_w, device):
    H, W = img_rgb.shape[:2]
    input_t = transforms.ToTensor()(
        pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    ).unsqueeze(0).to(device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        features = encoder(input_t)
        outputs  = decoder(features)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp = F.interpolate(outputs[("disp", 0)], (H, W),
                         mode="bilinear", align_corners=False)
    disp_np = disp.squeeze().cpu().numpy()
    min_d, max_d = 1/100, 1/0.1
    return 1.0 / (min_d + (max_d - min_d) * disp_np), t_ms


def depth_to_pc(depth_mm, mask, fx=FX, fy=FY, cx=CX, cy=CY):
    H, W = depth_mm.shape
    uu, vv = np.meshgrid(np.arange(W), np.arange(H))
    Z = depth_mm[mask]
    return np.stack([(uu[mask]-cx)*Z/fx, (vv[mask]-cy)*Z/fy, Z], axis=1)


def chamfer(pc1, pc2, max_pts=50_000):
    if len(pc1) == 0 or len(pc2) == 0:
        return np.nan
    rng = np.random.default_rng(42)
    if len(pc1) > max_pts: pc1 = pc1[rng.choice(len(pc1), max_pts, replace=False)]
    if len(pc2) > max_pts: pc2 = pc2[rng.choice(len(pc2), max_pts, replace=False)]
    d1, _ = cKDTree(pc2).query(pc1)
    d2, _ = cKDTree(pc1).query(pc2)
    return float((d1.mean() + d2.mean()) / 2)


def specular_mask(img_rgb, pct=97, dil=15):
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:, :, 0].astype(np.float32)
    mask = (L >= np.percentile(L, pct)).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dil, dil))
    return cv2.dilate(mask, kernel).astype(bool)


def compute_metrics(img_orig, img_corr, depth_rel, gt_mm, cap_mm=150.0):
    valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < cap_mm)
    if valid.sum() == 0:
        return {k: np.nan for k in
                ["AbsRel","RMSE","Chamfer","PSNR","SSIM",
                 "AbsRel_spec","AbsRel_nospec","scale"]}
    scale = np.median(gt_mm[valid]) / (np.median(depth_rel[valid]) + 1e-8)
    pred  = depth_rel * scale
    d, gt = pred[valid], gt_mm[valid]

    spec   = specular_mask(img_orig)
    vs, vn = valid & spec, valid & ~spec

    return {
        "scale":         round(float(scale), 4),
        "AbsRel":        round(float(np.mean(np.abs(d-gt)/(gt+1e-8))), 4),
        "RMSE":          round(float(np.sqrt(np.mean((d-gt)**2))), 2),
        "Chamfer":       round(chamfer(
                             depth_to_pc(np.where(valid,pred,np.nan), valid),
                             depth_to_pc(np.where(valid,gt_mm,np.nan), valid)), 3),
        "PSNR":          round(float(psnr_fn(img_orig, img_corr, data_range=255)), 2),
        "SSIM":          round(float(ssim_fn(img_orig, img_corr,
                                             channel_axis=2, data_range=255)), 4),
        "AbsRel_spec":   round(float(np.mean(np.abs(pred[vs]-gt_mm[vs])
                                             /(gt_mm[vs]+1e-8))), 4) if vs.sum()>0 else np.nan,
        "AbsRel_nospec": round(float(np.mean(np.abs(pred[vn]-gt_mm[vn])
                                             /(gt_mm[vn]+1e-8))), 4) if vn.sum()>0 else np.nan,
    }


print("Pipeline de inferencia y métricas definido ✓")

---
## 8. Experimento: 4 métodos × 10 keyframes

Para cada combinación (método, keyframe):
1. Cargar imagen RGB y GT
2. Aplicar el método de enhancement
3. Inferir depth con Endo-Depth (3 runs, reportar mediana)
4. Calcular métricas completas

Total: **40 evaluaciones**.

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

results = []

for dataset_id, keyframe_id in tqdm(EVAL_KEYFRAMES, desc="Keyframes"):
    img_rgb, gt_mm = load_keyframe(SCARED_ROOT, dataset_id, keyframe_id)

    for corr_name, corr_fn in CORRECTIONS.items():

        # Tiempo de corrección
        t_corr0 = time.perf_counter()
        img_corr = corr_fn(img_rgb)
        t_corr_ms = (time.perf_counter() - t_corr0) * 1000

        # Inferencia (3 runs para tiempo estable)
        times_inf = []
        for _ in range(3):
            depth_rel, t_ms = predict_depth(
                img_corr, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE)
            times_inf.append(t_ms)
        t_inf_ms = float(np.median(times_inf))

        metrics = compute_metrics(img_rgb, img_corr, depth_rel, gt_mm, CAP_MM)

        results.append({
            "Dataset":        dataset_id,
            "Keyframe":       keyframe_id,
            "Método":         corr_name,
            "T_enhance_ms":   round(t_corr_ms, 1),
            "T_depth_ms":     round(t_inf_ms, 1),
            "T_total_ms":     round(t_corr_ms + t_inf_ms, 1),
            "FPS":            round(1000 / (t_corr_ms + t_inf_ms), 1),
            **metrics,
        })

        print(f"  {dataset_id}/{keyframe_id} | {corr_name:12s} | "
              f"AbsRel={metrics['AbsRel']:.4f} "
              f"RMSE={metrics['RMSE']:.1f}mm "
              f"spec={metrics['AbsRel_spec']:.4f} "
              f"t={t_corr_ms+t_inf_ms:.0f}ms")

df = pd.DataFrame(results)
df.to_csv(OUT_DIR / "avance5_results.csv", index=False)
print(f"\nExperimento completo ✓  — {len(df)} evaluaciones")

---
## 9. Resultados

### 9.1 Tabla resumen por método

In [ ]:
import pandas as pd
import numpy as np

numeric_cols = ["AbsRel","RMSE","Chamfer","PSNR","SSIM",
                "AbsRel_spec","AbsRel_nospec","T_enhance_ms","T_depth_ms","T_total_ms","FPS"]

summary = (
    df.groupby("Método")[numeric_cols]
    .mean()
    .round(4)
    .sort_values("AbsRel")
)

# Calcular mejora relativa vs. baseline (none)
if "none" in summary.index:
    baseline = summary.loc["none"]
    for col in ["AbsRel", "RMSE", "Chamfer", "AbsRel_spec"]:
        summary[f"Δ{col}_%"] = ((summary[col] - baseline[col]) / baseline[col] * 100).round(1)

print("═" * 90)
print("RESUMEN — Promedio sobre 10 keyframes (datasets 8–9)")
print("═" * 90)
print(summary[["AbsRel","RMSE","Chamfer",
               "AbsRel_spec","AbsRel_nospec",
               "ΔAbsRel_%","ΔAbsRel_spec_%",
               "FPS"]].to_string())
print("─" * 90)
print("Δ% negativo = mejora respecto al baseline (none)")

### 9.2 Visualización comparativa de métricas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods = list(summary.index)
colors  = {"none": "#888888", "retinex": "#2766CB", "endolmspec": "#E0A800"}
x = np.arange(len(methods))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Evaluación de Image Enhancement sobre Endo-Depth en SCARED\n"
             "(promedio datasets 8–9, 10 keyframes)",
             fontsize=14, fontweight="bold")

plots = [
    ("AbsRel",        "AbsRel (↓ mejor)",             "Geométrica — primaria"),
    ("RMSE",          "RMSE mm (↓ mejor)",             "Geométrica"),
    ("Chamfer",       "Chamfer Distance mm (↓ mejor)", "Geométrica 3D"),
    ("AbsRel_spec",   "AbsRel especular (↓ mejor)",    "Robustez — hipótesis"),
    ("AbsRel_nospec", "AbsRel no-especular (↓ mejor)", "Robustez — contexto"),
    ("FPS",           "FPS total (↑ mejor)",           "Viabilidad clínica"),
]

for ax, (col, label, cat) in zip(axes.ravel(), plots):
    vals  = [summary.loc[m, col] for m in methods]
    bars  = ax.bar(x, vals,
                   color=[colors.get(m, "#aaa") for m in methods],
                   edgecolor="white", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=11)
    ax.set_title(f"{label}\n[{cat}]", fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(OUT_DIR / "avance5_metricas_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

### 9.3 Análisis de robustez: especular vs. no especular

La hipótesis central del proyecto predice que la mejora de los métodos de enhancement se concentra en las **zonas especulares** — donde el modelo de depth estaba más confundido por sobreexposición. Si `ΔAbsRel_spec` es mayor que `ΔAbsRel_nospec`, la hipótesis se confirma.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Robustez a especulares: AbsRel en zona especular vs. tejido normal",
             fontsize=13, fontweight="bold")

w = 0.3
x = np.arange(len(methods))

for ax, (ds_id, kf_id) in zip(axes, [EVAL_KEYFRAMES[0], None]):
    if kf_id is not None:
        sub = df[(df["Dataset"]==ds_id) & (df["Keyframe"]==kf_id)]
        title = f"{ds_id}/{kf_id}"
    else:
        sub = df
        title = "Promedio — todos los keyframes"

    spec_vals   = [sub[sub["Método"]==m]["AbsRel_spec"].mean()   for m in methods]
    nospec_vals = [sub[sub["Método"]==m]["AbsRel_nospec"].mean() for m in methods]

    b1 = ax.bar(x - w/2, spec_vals,   w, label="Zona especular",
                color="#e74c3c", alpha=0.85)
    b2 = ax.bar(x + w/2, nospec_vals, w, label="Tejido normal",
                color="#2766CB", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=11)
    ax.set_ylabel("AbsRel")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    for bars, vals in [(b1, spec_vals), (b2, nospec_vals)]:
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                        f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "avance5_robustez_especular.png", dpi=150, bbox_inches="tight")
plt.show()

# Interpretación cuantitativa
if "none" in summary.index:
    print("\n── Reducción de AbsRel_spec vs. baseline ──")
    b_spec = summary.loc["none", "AbsRel_spec"]
    b_nosp = summary.loc["none", "AbsRel_nospec"]
    for m in methods:
        if m == "none": continue
        s = summary.loc[m, "AbsRel_spec"]
        n = summary.loc[m, "AbsRel_nospec"]
        print(f"  {m:12s}: spec {s:.4f} ({(b_spec-s)/b_spec*100:+.1f}%)  "
              f"nospec {n:.4f} ({(b_nosp-n)/b_nosp*100:+.1f}%)")

---
## 10. Visualización cualitativa

Comparativa visual para los keyframes representativos: imagen corregida, depth map predicho, y mapa de error absoluto.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

VIZ_KEYFRAMES = [EVAL_KEYFRAMES[0], EVAL_KEYFRAMES[5]]  # dataset_8/kf_0 y dataset_9/kf_0

for ds_id, kf_id in VIZ_KEYFRAMES:
    img_rgb, gt_mm = load_keyframe(SCARED_ROOT, ds_id, kf_id)
    vmax = np.nanpercentile(gt_mm, 98)
    n = len(CORRECTIONS)

    fig, axes = plt.subplots(3, n + 1, figsize=(5*(n+1), 12))

    for col, (corr_name, corr_fn) in enumerate(CORRECTIONS.items()):
        img_c = corr_fn(img_rgb)
        depth_rel, _ = predict_depth(
            img_c, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE)
        valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < CAP_MM)
        scale = np.median(gt_mm[valid]) / (np.median(depth_rel[valid]) + 1e-8)
        pred_mm = depth_rel * scale

        m = compute_metrics(img_rgb, img_c, depth_rel, gt_mm, CAP_MM)

        axes[0, col].imshow(img_c)
        axes[0, col].set_title(
            f"{corr_name}\nPSNR={m['PSNR']:.1f}dB  SSIM={m['SSIM']:.3f}", fontsize=9)
        axes[0, col].axis("off")

        im = axes[1, col].imshow(pred_mm, cmap="magma_r", vmin=0, vmax=vmax)
        axes[1, col].set_title(
            f"AbsRel={m['AbsRel']:.4f}\nRMSE={m['RMSE']:.1f}mm", fontsize=9)
        axes[1, col].axis("off")

        err = np.abs(pred_mm - gt_mm)
        err[~valid] = np.nan
        axes[2, col].imshow(err, cmap="hot", vmin=0,
                            vmax=np.nanpercentile(err, 95))
        axes[2, col].set_title(
            f"spec={m['AbsRel_spec']:.4f}\nnospec={m['AbsRel_nospec']:.4f}", fontsize=9)
        axes[2, col].axis("off")

    # Columna GT
    axes[0, -1].imshow(img_rgb)
    axes[0, -1].set_title("Original", fontsize=9)
    axes[0, -1].axis("off")
    axes[1, -1].imshow(gt_mm, cmap="magma_r", vmin=0, vmax=vmax)
    axes[1, -1].set_title("GT (luz estructurada)", fontsize=9)
    axes[1, -1].axis("off")
    axes[2, -1].imshow(specular_mask(img_rgb), cmap="Reds")
    axes[2, -1].set_title("Máscara especular", fontsize=9)
    axes[2, -1].axis("off")

    plt.colorbar(im, ax=axes[1, :], label="mm", shrink=0.4, pad=0.01)
    plt.suptitle(f"Endo-Depth — {ds_id}/{kf_id}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"avance5_viz_{ds_id}_{kf_id}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

---
## 11. Conclusiones

### Hallazgos principales

*(se completa después de correr el experimento)*

| Método | AbsRel | RMSE | AbsRel_spec | FPS | Veredicto |
|---|---|---|---|---|---|
| None (baseline) | — | — | — | — | Referencia |
| Retinex SSR | — | — | — | — | |
| EndoLMSPEC | — | — | — | — | |
| IAT | — | — | — | — | |

### Comparativa de enfoques

| | **Retinex SSR** | **EndoLMSPEC** | **IAT** |
|---|---|---|---|
| **Tipo** | Clásico (sin aprendizaje) | U-Net + Laplaciana | Transformer local+global |
| **Entrenamiento** | No requiere | Endo4IE | MIT-FiveK / Exposure |
| **Parámetros** | 0 | ~2M | ~90K |
| **Resolución** | Cualquiera | Cualquiera | Cualquiera |
| **Costo típico** | ~5ms | ~120ms | ~30ms |

### Implicaciones clínicas

La viabilidad en tiempo real requiere ≥25 FPS (≤40ms). El tiempo total incluye enhancement + inferencia de depth (~25ms con Endo-Depth en L4).

## Referencias

- García-Vega, A., et al. (2022). Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging. *arXiv:2210.15033*.
- Afifi, M., et al. (2021). Learning multi-scale photo exposure correction. *CVPR 2021*.
- Wang, T., Zhang, K., Shen, T., Luo, W., Stenger, B., & Lu, T. (2022). Ultra-High-Definition Low-Light Image Enhancement: A Benchmark and Transformer-Based Method. *AAAI 2022*.
- Recasens, D., et al. (2021). Endo-Depth-and-Motion. *arXiv:2103.16525*.
- Rahman, Z., Jobson, D. J., & Woodell, G. A. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1), 100–110. https://doi.org/10.1117/1.1636183
- Land, E. H., & McCann, J. J. (1971). Lightness and retinex theory. *JOSA*, 61(1), 1–11.
- Allan, M., et al. (2021). Stereo Correspondence and Reconstruction of Endoscopic Data Challenge. *arXiv:2101.01133*.